In [ ]:
!pip install pysam

In [ ]:
import pysam
import pandas as pd
import statistics
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import rankdata
from scipy.special import ndtri

In [ ]:
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa'
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai'
fasta = pysam.FastaFile('GRCh38_full_analysis_set_plus_decoy_hla.fa')

In [ ]:
genome = 2294489308
opportunties = {'Total': 1, 'C>A': 907147366/genome, 'C>G': 907147366/genome, 'C>T': (907147366-39815717)/genome, 'CpG>TpG': 39815717/genome,
                'T>A' : (genome-907147366)/genome, 'T>C': (genome-907147366)/genome, 'T>G': (genome-907147366)/genome}

In [ ]:
def is_tp_ukb(mut, tp):
    
    chrom = int(mut.split(':')[0][3:])
    pos = int(mut.split(':')[1])
    ref = mut.split(':')[2]
    alt = mut.split(':')[3]
    
    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}
    
    cont = fasta.fetch(f'chr{chrom}', pos-2, pos+1)
    mut = f'{cont[0]}[{ref}>{alt}]{cont[2]}'

    if ref == 'A' or ref == 'G':
        ref = comp[ref]
        alt = comp[alt]
        cont = f'{comp[cont[0]]}{comp[cont[1]]}{comp[cont[2]]}'
        cont = cont[::-1]
        mut = (comp[mut[6]]+mut[1]+comp[mut[2]]+mut[3]+ 
               comp[mut[4]]+mut[5]+comp[mut[0]])

    if tp == 'CpG>TpG':
        return 1 if (mut[2] == 'C' and mut[4] == 'T' and mut[6] == 'G') else 0
    elif tp == 'C>T':
        return 1 if (mut[2] == 'C' and mut[4] == 'T' and mut[6] != 'G') else 0
    return 1 if (mut[2] == tp[0] and mut[4] == tp[2]) else 0

In [ ]:
def count_tp_ukb(muts, tp):
    
    if tp == 'Total':
        return len(muts)
    return sum(is_tp_ukb(mut, tp) for mut in muts)

# UKB

## Trios

In [ ]:
trios_ukb_dnms = {}

trios_ukb_dnms_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info.txt', sep = '\t')

for _, row in trios_ukb_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = int(ind_idx.split('_')[0])
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind not in trios_ukb_dnms:
        trios_ukb_dnms[ind] = set()
    trios_ukb_dnms[ind].add(f'chr{chrom}:{pos}:{ref}:{alt}')

In [ ]:
trios_ukb_rates = {'Total': {}, 'C>A': {}, 'C>G': {}, 'C>T': {}, 'CpG>TpG': {}, 'T>A': {}, 'T>C': {}, 'T>G': {}}

for ind in trios_ukb_dnms:
    for tp in trios_ukb_rates:
        trios_ukb_rates[tp][ind] = count_tp_ukb(trios_ukb_dnms[ind], tp)/(genome*2*opportunties[tp])

## Sibs

In [ ]:
sib_ukb_dnms = {}

sibs_ukb_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info.txt', sep = '\t')

for _, row in sibs_ukb_dnms_df.iterrows():
    
    ind_idx = row['IID']
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind_idx not in sib_ukb_dnms:
        sib_ukb_dnms[ind_idx] = set()
    sib_ukb_dnms[ind_idx].add(f'chr{chrom}:{pos}:{ref}:{alt}')

In [ ]:
sibs_ukb_dnms = {}

sibs_ukb_df = pd.read_csv('/mnt/project/DNM/sibs_pairs.txt', sep = ' ', header = None)

for i, row in sibs_ukb_df.iterrows():
    
    sib1 = int(row[0])
    sib2 = int(row[1])
    
    b = int(i/100)
    p = i%100
            
    sib1_idx = f'{sib1}_b{b}_p{p}'
    sib2_idx = f'{sib2}_b{b}_p{p}'
    
    if sib1_idx not in sib_ukb_dnms and sib2_idx not in sib_ukb_dnms:
        continue

    elif sib1_idx not in sib_ukb_dnms:
        sib_ukb_dnms[sib1_idx] = set()
    elif sib2_idx not in sib_ukb_dnms:
        sib_ukb_dnms[sib2_idx] = set()
    
    sibs_ukb_dnms[(sib1, sib2)] = sib_ukb_dnms[sib1_idx].union(sib_ukb_dnms[sib2_idx])

In [ ]:
ibd2_ukb_filtered = {}

ibd2_ukb_df = pd.read_csv('ukb_ibd2_giab_removed.tsv', sep = '\t')

for _, row in ibd2_ukb_df.iterrows():
    sib1 = int(row['pair'].split('_')[0])
    sib2 = int(row['pair'].split('_')[1])
    ibd2_ukb_filtered[(sib1, sib2)] = row['length_filter']

In [ ]:
sibs_ukb_rates = {'Total': {}, 'C>A': {}, 'C>G': {}, 'C>T': {}, 'CpG>TpG': {}, 'T>A': {}, 'T>C': {}, 'T>G': {}}

for pair in sibs_ukb_dnms:
    for tp in sibs_ukb_rates:
        sibs_ukb_rates[tp][pair] = count_tp_ukb(sibs_ukb_dnms[pair], tp)/(ibd2_ukb_filtered[pair]*4*opportunties[tp])

## Families

In [ ]:
fams = {}
ind_fam = {}

fams_df = pd.read_csv('ukb_families.csv')

for _, row in fams_df.iterrows():
    i = row['ID']
    fams[i] = [int(ind) for ind in row['INDS'].split(',')]
    for ind in fams[i]:
        ind_fam[ind] = i

In [ ]:
fam_inds = {i: [] for i in fams}

for ind in trios_ukb_rates['Total']:
    if ind in ind_fam:
        i = ind_fam[ind]
        fam_inds[i].append(ind)

for sib1, sib2 in sibs_ukb_rates['Total']:
    if sib1 in trios_ukb_rates['Total'] and sib2 in trios_ukb_rates['Total']:
        continue
    if sib1 in ind_fam and sib2 in ind_fam:
        i = ind_fam[sib1]
        fam_inds[i].append((sib1, sib2))

In [ ]:
trio_cnt = (0, 0)
sib_cnt = (0, 0)

for i in fam_inds:

    if isinstance(fam_inds[i][0], int):
        trio_cnt = (trio_cnt[0]+len(fam_inds[i]), trio_cnt[1]+1)
    elif isinstance(fam_inds[i][0], tuple):
        sib_cnt = (sib_cnt[0]+len(fam_inds[i]), sib_cnt[1]+1)

print(trio_cnt, sib_cnt)

In [ ]:
len(fam_inds)

In [ ]:
fams_trios_ukb_rates = {'Total': {}, 'C>A': {}, 'C>G': {}, 'C>T': {}, 'CpG>TpG': {}, 'T>A': {}, 'T>C': {}, 'T>G': {}}

for i in fams:
    if isinstance(fam_inds[i][0], int):
        for tp in fams_trios_ukb_rates:
            fam_trios_ukb_rates = [trios_ukb_rates[tp][ind] for ind in fam_inds[i]]
            fams_trios_ukb_rates[tp][i] = sum(fam_trios_ukb_rates)/len(fam_trios_ukb_rates)

In [ ]:
for tp in fams_trios_ukb_rates:
    
    print(f'{min(fams_trios_ukb_rates[tp].values()):.4e} {max(fams_trios_ukb_rates[tp].values()):.4e} {statistics.mean(fams_trios_ukb_rates[tp].values()):.4e} {statistics.median(fams_trios_ukb_rates[tp].values()):.4e} {statistics.variance(fams_trios_ukb_rates[tp].values()):.4e} {len(fams_trios_ukb_rates[tp].values())}')

    plt.hist(fams_trios_ukb_rates[tp].values(), bins = 10)
    plt.xlabel('De novo mutation rate (UKB trios)' if tp == 'Total' else f'De novo {tp} rate (UKB trios)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
    plt.clf()

In [ ]:
fams_sibs_ukb_rates = {'Total': {}, 'C>A': {}, 'C>G': {}, 'C>T': {}, 'CpG>TpG': {}, 'T>A': {}, 'T>C': {}, 'T>G': {}}

for i in fams:
    if isinstance(fam_inds[i][0], tuple):
        for tp in fams_sibs_ukb_rates:
            fam_sibs_ukb_rates = [sibs_ukb_rates[tp][pair] for pair in fam_inds[i]]
            fams_sibs_ukb_rates[tp][i] = sum(fam_sibs_ukb_rates)/len(fam_sibs_ukb_rates)

In [ ]:
for tp in fams_sibs_ukb_rates:

    print(f'{min(fams_sibs_ukb_rates[tp].values()):.4e} {max(fams_sibs_ukb_rates[tp].values()):.4e} {statistics.mean(fams_sibs_ukb_rates[tp].values()):.4e} {statistics.median(fams_sibs_ukb_rates[tp].values()):.4e} {statistics.variance(fams_sibs_ukb_rates[tp].values()):.4e} {len(fams_sibs_ukb_rates[tp].values())}')

    plt.hist(fams_sibs_ukb_rates[tp].values(), bins = 15)
    plt.xlabel('De novo mutation rate (UKB sibs)' if tp == 'Total' else f'De novo {tp} rate (UKB sibs)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
    plt.clf()

In [ ]:
for tp in fams_trios_ukb_rates:
    
    if tp == 'Total':
        f = 'ukb_fams_rates.txt'
    elif tp == 'C>A':
        f = 'ukb_fams_c_a_rates.txt'
    elif tp == 'C>G':
        f = 'ukb_fams_c_g_rates.txt'
    elif tp == 'C>T':
        f = 'ukb_fams_c_t_rates.txt'
    elif tp == 'CpG>TpG':
        f = 'ukb_fams_cpg_tpg_rates.txt'
    elif tp == 'T>A':
        f = 'ukb_fams_t_a_rates.txt'
    elif tp == 'T>C':
        f = 'ukb_fams_t_c_rates.txt'
    elif tp == 'T>G':
        f = 'ukb_fams_t_g_rates.txt'

    with open(f, 'w') as f:

        for i in fams_trios_ukb_rates[tp]:
            fam = '_'.join(map(str, fams[i]))
            f.write(f'{fam} {fams_trios_ukb_rates[tp][i]}\n')

        for i in fams_sibs_ukb_rates[tp]:
            fam = '_'.join(map(str, fams[i]))
            f.write(f'{fam} {fams_sibs_ukb_rates[tp][i]}\n')

In [ ]:
for tp in fams_trios_ukb_rates:
    print(tp, 'trio', sum(r == 0 for r in fams_trios_ukb_rates[tp].values()) / len(fams_trios_ukb_rates[tp].values()),
          'sib', sum(r == 0 for r in fams_sibs_ukb_rates[tp].values()) / len(fams_sibs_ukb_rates[tp].values()))